In this notebook, we will scrape the CUNY Fall 2021 Academic Calendar from its webpage, using BeautifulSoup, and convert it into a pandas DataFrame. To start, we will import the necessary packages.

In [32]:
# Fork of the requests library that uses the same syntax but allows for the impersonation of a browser.
# This is required by websites that have certain anti-bot measures in place.
from curl_cffi import requests
# BeautifulSoup will allow us to easily parse the HTML we get from the webpage.
from bs4 import BeautifulSoup
# pandas will allow us to use DataFrames, which are an essential tool in scripting for data science.
import pandas as pd

We will have to convert dates and weekdays written in English into a standard Python format. For this, it is helpful to store the following lists to easily convert to and from names and numbers.

In [33]:
weekdays = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
months = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]

Next, we specify the URL and perform the GET request, while impersonating an instance of the Google Chrome browser.

In [34]:
url = 'https://www.ccny.cuny.edu/registrar/fall'
r = requests.get(url, impersonate="chrome")

We convert the text of the response from our GET request into a BeautifulSoup object. This will allow us to more easily find specific elements. We can call the `prettify()` method to get a string containing the HTML code in a more readable ("pretty") format.

In [35]:
soup = BeautifulSoup(r.text, "html.parser")
print(soup.prettify())

<!DOCTYPE html>
<html dir="ltr" lang="en" prefix="content: http://purl.org/rss/1.0/modules/content/  dc: http://purl.org/dc/terms/  foaf: http://xmlns.com/foaf/0.1/  og: http://ogp.me/ns#  rdfs: http://www.w3.org/2000/01/rdf-schema#  schema: http://schema.org/  sioc: http://rdfs.org/sioc/ns#  sioct: http://rdfs.org/sioc/types#  skos: http://www.w3.org/2004/02/skos/core#  xsd: http://www.w3.org/2001/XMLSchema# ">
 <head>
  <meta charset="utf-8"/>
  <meta content="The City College of New York" name="geo.placename"/>
  <meta content="40.8200471,-73.9492724" name="geo.position"/>
  <meta content="40.8200471,-73.9492724" name="icbm"/>
  <meta content="US-NY" name="geo.region"/>
  <meta content="index, follow" name="robots"/>
  <link href="https://www.ccny.cuny.edu/registrar/fall" rel="canonical"/>
  <meta content="The City College of New York" name="author"/>
  <meta content="Drupal 9" name="generator"/>
  <meta content="The City College of New Yoprk" property="og:site_name"/>
  <meta conte

We initialize an empty list of lists to store our data, which we will convert into a DataFrame later.

In [36]:
data = []

Analysis of the HTML reveals that each row of data we want is stored in a `<tr>` element, and the value of each column within each row is stored in a `<td>` element. We can use BeautifulSoup's `find_all` method to get all elements of a specific type.

Afterwards, we need to parse the text we find within these elements into a consistent format, the process of which is documented in comments in the code.

In [37]:
rows = soup.find_all("tr")
del rows[0] # garbage data
for row in rows:
    cols = [" ".join(td.text.split()) for td in row.find_all("td")] # normalizes whitespace for each column
    # Parse the dates into a consistent format.
    if "," in cols[0]:
        # If the date contains a comma, this means the year is specified.
        year = int(cols[0][-4:])
    else:
        # Otherwise, the year is 2021.
        year = 2021
    month = months.index(cols[0].split()[0])+1
    # If there's a hyphen, it means there are multiple dates.
    if "-" in cols[0]:
        lday = int(cols[0].split()[1])
        rday = int(cols[0].split()[3])
        lwday = weekdays.index(cols[1].split()[0])
        rwday = weekdays.index(cols[1].split()[2]) # We don't need this, but it looks nice.
        # Iterate through the range of dates.
        for i in range(0, rday-lday+1):
            day = lday+i
            wday = (lwday+i)%7
            # Copy the row.
            tmpcols = cols[:]
            # Apply consistent formatting to year, month, day numbers.
            tmpcols[0] = f"{year:04d}-{month:02d}-{day:02d}"
            # Convert weekday indices into weekday names.
            tmpcols[1] = weekdays[wday]
            # Append the row to the data.
            data.append(tmpcols)
    # Otherwise, it can be treated as a single row.
    else:
        day = cols[0].split()[1]
        # In cases where the year is specified with a comma, the comma needs to be removed.
        if "," in day:
            day = day.replace(",","")
        day = int(day)
        # Apply consistent formatting to year, month, day numbers.
        cols[0] = f"{year:04d}-{month:02d}-{day:02d}"
        # Append the row to the data.
        data.append(cols)

Now we perform some manual parsing on the values in the columns. This is slightly tricky, as not only do we have to convert from an unstandardized human-readable date format to a consistent Python date format, but some dates are expressed as ranges, which need to be divided into separate columns.

Before moving on, let's check that the data list is correctly populated.

In [38]:
print(data)

[['2021-08-01', 'Sunday', 'Application for degree for January and February 2022 begins'], ['2021-08-18', 'Wednesday', 'Last day to apply for Study Abroad'], ['2021-08-24', 'Tuesday', 'Last day of Registration; Last day to file ePermit for the Fall 2021; Last day to drop classes for 100% tuition refund;'], ['2021-08-25', 'Wednesday', 'Start of Fall Term; Classes begin; Initial Registration Appeals begin;'], ['2021-08-25', 'Wednesday', 'Change of program period; late fees apply'], ['2021-08-26', 'Thursday', 'Change of program period; late fees apply'], ['2021-08-27', 'Friday', 'Change of program period; late fees apply'], ['2021-08-28', 'Saturday', 'Change of program period; late fees apply'], ['2021-08-29', 'Sunday', 'Change of program period; late fees apply'], ['2021-08-30', 'Monday', 'Change of program period; late fees apply'], ['2021-08-31', 'Tuesday', 'Change of program period; late fees apply'], ['2021-08-26', 'Thursday', 'Last day for Independent Study'], ['2021-08-28', 'Saturda

Now that we have a list of lists containing all the data we want, we can convert it into a DataFrame.

In [39]:
df = pd.DataFrame(data, columns=["date", "day_of_week", "explanation"])
print(df.head())

         date day_of_week                                        explanation
0  2021-08-01      Sunday  Application for degree for January and Februar...
1  2021-08-18   Wednesday                 Last day to apply for Study Abroad
2  2021-08-24     Tuesday  Last day of Registration; Last day to file ePe...
3  2021-08-25   Wednesday  Start of Fall Term; Classes begin; Initial Reg...
4  2021-08-25   Wednesday          Change of program period; late fees apply


The above DataFrame has an index consisting of meaningless integers, which we don't need. We would prefer to index the DataFrame by dates.

In [40]:
# Index the DataFrame by date.
df_ind = df.set_index("date")
# Convert index into proper datetime objects.
df_ind.index = pd.to_datetime(df_ind.index)
# Sort DataFrame.
df_ind = df_ind.sort_index()
print(df_ind.head())

           day_of_week                                        explanation
date                                                                     
2021-08-01      Sunday  Application for degree for January and Februar...
2021-08-18   Wednesday                 Last day to apply for Study Abroad
2021-08-24     Tuesday  Last day of Registration; Last day to file ePe...
2021-08-25   Wednesday  Start of Fall Term; Classes begin; Initial Reg...
2021-08-25   Wednesday          Change of program period; late fees apply


We can verify that the date conversion worked by using specialized range operations for dates. In this example, we get all of the data for dates in the month of September, 2021.

In [41]:
print(df_ind["2021-09":"2021-09"])

           day_of_week                                        explanation
date                                                                     
2021-09-01   Wednesday  Verification of Enrollment rosters available t...
2021-09-03      Friday                               No classes scheduled
2021-09-04    Saturday                               No classes scheduled
2021-09-05      Sunday                               No classes scheduled
2021-09-06      Monday                               No classes scheduled
2021-09-06      Monday   College Closed; Last day for 50% tuition refund;
2021-09-07     Tuesday                               No classes scheduled
2021-09-08   Wednesday                               No classes scheduled
2021-09-09    Thursday                               Freshman Convocation
2021-09-14     Tuesday  Last day for 25% tuition refund; Census date; ...
2021-09-15   Wednesday  Assignment of 'WN' grades for non-attendance; ...
2021-09-15   Wednesday                